In [1]:
!pip install transformers datasets torch scikit-learn -q

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer

In [3]:
model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_b = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

In [5]:
print("Loading the 35,222 sample Master Dataset...")
from sklearn.metrics import accuracy_score, f1_score # <--- Make sure f1_score is here
df_master = pd.read_csv('/kaggle/input/datasets/janithviranga/sri-lankan-act-data/LAWNOVA_MASTER_TRAINING_DATA.csv')

# Convert to HF Dataset
dataset = Dataset.from_pandas(df_master).map(
    lambda x: tokenizer(x["text"], padding="max_length", truncation=True, max_length=128), 
    batched=True
).train_test_split(test_size=0.1, seed=42) # 10% for validation (3,500 rows)

# ==========================================
# STEP 3: Setup Metrics
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {"accuracy": acc, "f1": f1}

# ==========================================
# STEP 4: Training Arguments (Optimized for 35k)
# ==========================================
training_args = TrainingArguments(
    output_dir="./results_model_b",
    num_train_epochs=3,                 # 3 epochs is plenty for 35k samples
    per_device_train_batch_size=32,     # Higher batch size for better stability
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer_b = Trainer(
    model=model_b,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics
)

print("\n--- Starting Deep Training for LAWNOVA LAW EXPERT ---")
trainer_b.train()



Loading the 35,222 sample Master Dataset...


Map:   0%|          | 0/35222 [00:00<?, ? examples/s]


--- Starting Deep Training for LAWNOVA LAW EXPERT ---


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.058948,0.995458,0.995458
2,0.021353,0.046551,0.996026,0.996026
3,0.010113,0.042484,0.996594,0.996594


...

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

NameError: name 'shutil' is not defined

In [6]:
import shutil
from google.colab import files

In [8]:
from IPython.display import FileLink
model_b.save_pretrained("/kaggle/working/LAWNOVA_LAW_EXPERT")
tokenizer.save_pretrained("/kaggle/working/LAWNOVA_LAW_EXPERT")

shutil.make_archive('LAWNOVA_MODEL_B_35K', 'zip', '/kaggle/working/LAWNOVA_LAW_EXPERT')
print("\nTraining Complete! Download your 35k Champion Model B here:")
FileLink(r'LAWNOVA_MODEL_B_35K.zip')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training Complete! Download your 35k Champion Model B here:


/kaggle/working/LAWNOVA_MODEL_B_35K.zip

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Load your Model B from the working directory
model_path = "/kaggle/working/LAWNOVA_LAW_EXPERT"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

# 2. Define "Trick" Test Cases
# These are designed to see if the model can tell the difference 
# between someone TALKING about a law (Fact) and the LAW itself (Statute).
test_queries = [
    # True Law (Statutes)
    "Whoever commits murder shall be punished with death.",
    "The burden of proof as to any particular fact lies on that person who wishes the court to believe in its existence.",
    "Section 300: Whoever does any act with such intention or knowledge that if he by that act caused death he would be guilty of murder.",
    
    # Trick Cases (Facts that look like Law)
    "The witness stated that he saw the defendant violating Section 300 of the Penal Code.", 
    "According to the Fiscal report, the land was sold under a mortgage bond case.",
    "My lawyer told me that I should plead not guilty under the Evidence Ordinance.",
    
    # Pure Facts
    "The plaintiff was ejected from Lot 1 on the 10th of May 2024.",
    "A large crowd gathered outside the house when the police arrived with the warrant."
]

print(f"{'Text Snippet':<80} | {'Prediction':<10} | {'Confidence'}")
print("-" * 110)

for text in test_queries:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        prediction = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][prediction].item()
        
    label_map = {1: "LAW (1)", 0: "FACT (0)"}
    print(f"{text[:78]:<80} | {label_map[prediction]:<10} | {confidence:.2%}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Text Snippet                                                                     | Prediction | Confidence
--------------------------------------------------------------------------------------------------------------
Whoever commits murder shall be punished with death.                             | FACT (0)   | 97.45%
The burden of proof as to any particular fact lies on that person who wishes t   | FACT (0)   | 99.88%
Section 300: Whoever does any act with such intention or knowledge that if he    | LAW (1)    | 100.00%
The witness stated that he saw the defendant violating Section 300 of the Pena   | LAW (1)    | 100.00%
According to the Fiscal report, the land was sold under a mortgage bond case.    | FACT (0)   | 99.93%
My lawyer told me that I should plead not guilty under the Evidence Ordinance.   | LAW (1)    | 100.00%
The plaintiff was ejected from Lot 1 on the 10th of May 2024.                    | FACT (0)   | 100.00%
A large crowd gathered outside the house when the police 

In [12]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Create the "Anti-Overfit" Dataset
# We intentionally mix keywords into the 'wrong' categories to force the model to read context.
correction_data = [
    # --- Facts with Legal Keywords (Label 0) ---
    {"text": "The witness mentioned Section 300 while describing the fight.", "label": 0},
    {"text": "My lawyer told me that Section 4 of the Act was violated.", "label": 0},
    {"text": "The police sergeant said he filed the report under the Penal Code.", "label": 0},
    {"text": "The defendant admitted that he knew about the Evidence Ordinance.", "label": 0},
    {"text": "According to the surveyor, the boundary was marked in the deed.", "label": 0},
    
    # --- Law without specific 'Section' numbers (Label 1) ---
    {"text": "Whoever commits murder shall be punished with death.", "label": 1},
    {"text": "The burden of proof lies on the person who wishes the court to believe a fact.", "label": 1},
    {"text": "All evidence must be relevant to the facts in issue to be admissible.", "label": 1},
    {"text": "A person is presumed innocent until proven guilty beyond reasonable doubt.", "label": 1}
]

df_corrections = pd.DataFrame(correction_data)

df_master = pd.read_csv('/kaggle/input/datasets/janithviranga/sri-lankan-act-data/LAWNOVA_MASTER_TRAINING_DATA.csv')
df_combined = pd.concat([df_corrections, df_master.sample(500, random_state=42)]).sample(frac=1)

# 3. Load your 99% Model B
model_path = "/kaggle/working/LAWNOVA_LAW_EXPERT"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# 4. Fine-Tune for 1-2 Epochs (Very low learning rate)
dataset = Dataset.from_pandas(df_combined).map(
    lambda x: tokenizer(x["text"], padding="max_length", truncation=True, max_length=128), batched=True
)

training_args = TrainingArguments(
    output_dir="./LAWNOVA_LAW_EXPERT_FIXED",
    num_train_epochs=2,
    learning_rate=5e-6, # Extremely low to avoid destroying the 99% foundation
    per_device_train_batch_size=8,
    report_to="none"
)

trainer = Trainer(model=model, args=training_args, train_dataset=dataset)
trainer.train()

# 5. Save the New "Fixed" Champion
model.save_pretrained("/kaggle/working/LAWNOVA_LAW_EXPERT_FIXED")
tokenizer.save_pretrained("/kaggle/working/LAWNOVA_LAW_EXPERT_FIXED")
print("Model B has been successfully re-balanced!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/509 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model B has been successfully re-balanced!


In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Load the FIXED Model B
model_path = "/kaggle/working/LAWNOVA_LAW_EXPERT_FIXED"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

# 2. Re-run the problematic queries
test_queries = [
    # Test 1: Can it recognize Law WITHOUT a section number now?
    "Whoever commits murder shall be punished with death.",
    
    # Test 2: Can it recognize a Fact even WITH a section number?
    "The witness stated that he saw the defendant violating Section 300 of the Penal Code.", 
    
    # Test 3: Can it recognize the burden of proof as Law?
    "The burden of proof as to any particular fact lies on that person who wishes the court to believe in its existence.",
    
    # Test 4: General Legal Scenario (Should be Fact)
    "My lawyer told me that I should plead not guilty under the Evidence Ordinance.",
    
    # Test 5: Standard Fact
    "The plaintiff was ejected from Lot 1 on the 10th of May 2024."
]

print(f"{'Text Snippet':<80} | {'Prediction':<10} | {'Confidence'}")
print("-" * 110)

for text in test_queries:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        prediction = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][prediction].item()
        
    label_map = {1: "LAW (1)", 0: "FACT (0)"}
    print(f"{text[:78]:<80} | {label_map[prediction]:<10} | {confidence:.2%}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Text Snippet                                                                     | Prediction | Confidence
--------------------------------------------------------------------------------------------------------------
Whoever commits murder shall be punished with death.                             | LAW (1)    | 91.09%
The witness stated that he saw the defendant violating Section 300 of the Pena   | LAW (1)    | 100.00%
The burden of proof as to any particular fact lies on that person who wishes t   | FACT (0)   | 86.79%
My lawyer told me that I should plead not guilty under the Evidence Ordinance.   | LAW (1)    | 100.00%
The plaintiff was ejected from Lot 1 on the 10th of May 2024.                    | FACT (0)   | 100.00%


In [15]:
import shutil
import os
from IPython.display import FileLink

# 1. Define the path where your trained model is saved
# If you used the 'FIXED' version, use that path
model_path = '/kaggle/working/LAWNOVA_LAW_EXPERT_FIXED'
zip_name = 'LAWNOVA_MODEL_B_FINAL'

# 2. Check if the directory exists to avoid errors
if os.path.exists(model_path):
    shutil.make_archive(zip_name, 'zip', model_path)
    print(f"Successfully zipped {model_path}")
    display(FileLink(f'{zip_name}.zip'))
else:
    print("Error: Model directory not found. Please check your path.")

Successfully zipped /kaggle/working/LAWNOVA_LAW_EXPERT_FIXED


/kaggle/working/LAWNOVA_MODEL_B_FINAL.zip